# Marine Monitoring Measurements from Estuaries and Coast of the Basque Country, 1995–2014 Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.8jkp-vqbe/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in Croissant datasets are referenced by their `@id`. Let's inspect what record sets and fields are available.

In [ ]:
# List available record sets and fields for overview

# Get all record sets from the metadata
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if hasattr(rs, '@id'):
            record_sets.append(rs)
            print(f"RecordSet @id: {rs['@id']}")
            if hasattr(rs, 'field'):
                print("  Fields:")
                for f in rs['field']:
                    print(f"    Field @id: {f['@id']}, name: {f.get('name', 'N/A')}")
else:
    print("No record sets found in metadata.")

# If there are no record sets in metadata (possible in this FAIR2 package), show distributions
if not record_sets and hasattr(metadata, 'distribution'):
    print("Distributions found (may contain records):")
    for dist in metadata.distribution:
        if hasattr(dist, '@id'):
            print(f"Distribution @id: {dist['@id']}")

## 3. Data Extraction
Load data from a specific record set or distribution into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

In [ ]:
# If no record sets, explore distributions directly
# Store available record set IDs or distribution IDs
record_sets_ids = [rs['@id'] for rs in record_sets] if record_sets else []
distribution_ids = [dist['@id'] for dist in metadata.distribution] if hasattr(metadata, 'distribution') else []

dataframes = {}

# Try using record sets first
if record_sets_ids:
    for record_set_id in record_sets_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    # Show columns of the first record set
    show_id = record_sets_ids[0]
    print(f"Columns in record set {show_id}:", dataframes[show_id].columns.tolist())
    display(dataframes[show_id].head())
elif distribution_ids:
    # If distributions are available, try to load their data
    for distribution_id in distribution_ids:
        print(f"Attempting to load records from distribution {distribution_id}")
        try:
            records = list(dataset.records(record_set=distribution_id))
            df = pd.DataFrame(records)
            if not df.empty:
                dataframes[distribution_id] = df
                print(f"Columns in distribution {distribution_id}:", df.columns.tolist())
                display(df.head())
        except Exception as e:
            print(f"Could not load records from {distribution_id}: {e}")
else:
    print("No record sets or distributions available for data extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes operations like removing outliers or grouping data by attributes, to prepare it for further analysis.

We'll demonstrate using an available DataFrame. Fields and columns are referenced by their `@id`.

In [ ]:
# Choose a DataFrame and fields for EDA
import numpy as np
from matplotlib import pyplot as plt

# Pick the first DataFrame loaded, if available
if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    print(f"Using DataFrame from: {df_key}")

    # Find numeric fields (by inspecting columns)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Selected numeric field: {numeric_field}")

        # Filter records above a threshold
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field]).all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].head()])

        # Grouping by another field if available
        group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the selected numeric field and, if possible, visualize relationships grouped by a categorical field.

In [ ]:
if dataframes:
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        plt.figure(figsize=(8,4))
        plt.hist(df[numeric_field].dropna(), bins=20, color='skyblue', edgecolor='black')
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()

        # If group_field exists, plot average numeric_field per group
        group_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        if group_fields:
            group_field = group_fields[0]
            grouped = df.groupby(group_field)[numeric_field].mean()
            grouped.plot(kind='bar', figsize=(10,4), color='teal')
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.ylabel(f"Mean {numeric_field}")
            plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("No DataFrame available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using the Croissant schema, the dataset was loaded and its structure explored.
- Data was extracted and filtered based on numeric criteria, normalized, and grouped for summary statistics.
- Distributions and group relationships were visualized, enabling understanding of the marine monitoring measurements.
- For detailed field and record set information, always refer to their unique `@id` as provided in the Croissant schema.

Further exploration can include temporal trend analysis, location-based grouping, or deep dives into ecological relationships using additional fields within the dataset.